In [2]:
from insightface.app import FaceAnalysis
from noisereduce import reduce_noise
from dotenv import load_dotenv
from datetime import datetime
from scipy.io import wavfile
from openai import OpenAI


import os
import io
import cv2
import vosk
import time
import json
import uuid
import wave
import librosa
import tempfile
import whisper
import pyttsx3
import pyaudio
import requests
import warnings
import numpy as np
import onnxruntime
import insightface
import speech_recognition as sr   
import matplotlib.pyplot as plt
 
onnxruntime.set_default_logger_severity(3)
warnings.filterwarnings('ignore')

# 1. Face Verification

In [ ]:
def verify_the_face(SIMILARITY_THRESHOLD=40,
                   camera_input_index=0,
                   weights="buffalo_l",
                   root="C:\\Users\\laste\\Desktop\\Scene Understanding\\face_verification_models\\.insightface\\",
                   reference_image="sample.jpg",
                   use_gpu=True):
    
    # Initialize model
    app = FaceAnalysis(name=weights, root=root)
    app.prepare(ctx_id=0 if use_gpu else -1, det_size=(640, 640))

    # Load reference face
    your_face_img = cv2.imread(reference_image)
    if your_face_img is None:
        print(f"Error: Reference image not found at {reference_image}")
        return 0
        
    your_face = app.get(your_face_img)
    if not your_face:
        print("Error: No face detected in reference image")
        return 0
    
    anchor_face_embedding = your_face[0].embedding
    permission_flag = 0
    found_you = False
    unknown_faces = set()

    cap = cv2.VideoCapture(camera_input_index)
    if not cap.isOpened():
        print(f"Error: Could not open camera {camera_input_index}")
        return 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            faces = app.get(frame)
            if not faces:
                cv2.imshow("Face Verification", frame)
                if cv2.waitKey(1) == ord('q'):
                    break
                continue
                
            current_frame_face_ids = set()
            
            # for face in faces:
            face = faces[0]
            max_sim = (face.embedding @ face.embedding.T).item()
            similarity = (anchor_face_embedding @ face.embedding.T).item()
            similarity = (similarity * 100) / max_sim 
            x1, y1, x2, y2 = face.bbox.astype(int)
            face_id = f"{x1}_{y1}_{x2}_{y2}"
            current_frame_face_ids.add(face_id)
                
            if similarity >= SIMILARITY_THRESHOLD and not found_you:

                found_you = True
                permission_flag = 1
                color = (0, 255, 0)  # Green
                                        
                # Draw green box and show for 3 seconds
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.imshow("Face Verification", frame)
                cv2.waitKey(2000)
                break
    
            else:
                # if it not a verified person
                color = (0, 0, 255) 
                timestamp = datetime.now().strftime("%m%d_%H%M%S")
                permission_flag = 0
                
                filename = f"Unknown_Person_M{timestamp[0:2]}_D{timestamp[2:4]}_H{timestamp[-6:-4]}_M{timestamp[-4:-2]}_S{timestamp[-2:]}.jpg"
                # print(filename)
                cv2.imwrite(filename, frame)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.imshow("Face Verification", frame)
                cv2.waitKey(2000)
                break

            if found_you:
                break
                
                
    finally:
        cap.release()
        cv2.destroyAllWindows()
    
    return permission_flag

# 2. Voice to Text Conversion (Vice Versa)

In [4]:
# A function to record voice using PyAudio
def start_recording(RECORD_SECONDS = 20):
    
    CHUNK = 1024
    FORAMT = pyaudio.paInt16
    CHANNELS = 1 
    # RATE = 44100
    RATE = 16000 # VOSK compatible sample_rate
    
    p = pyaudio.PyAudio()
    stream = p.open(format=FORAMT,
                    channels=CHANNELS,
                    rate = RATE,
                    input = True,
                    input_device_index=1,
                    frames_per_buffer=CHUNK)

    print("\nListening....\n")
    
    
    # Collecting Voice frames
    frames=[]
    for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
        data = stream.read(CHUNK)
        frames.append(data)

        # if keyboard.is_pressed("esc"):
        #     stop_recording()
        #     return
        # Nothing more than 30 seconds
        if len(frames) > int(RATE / CHUNK * RECORD_SECONDS):
            break
        

    stream.stop_stream()
    stream.close()
    p.terminate()
    print("\nThinking... \n")

    # Compress all the frames into a single byte
    audio_data = b''.join(frames)
    audio = sr.AudioData(audio_data, sample_rate=RATE, sample_width=2)
    
    return audio

In [ ]:
"""Google (Online Solution)"""
# # covert it into a text prompt
# def recognize_speech(audio_data):
#     r = sr.Recognizer()
#     try:
#         text = r.recognize_google(audio_data, language="fa")
#         print(f"Recognized Input Voice: {text}")
#         return text

#     except sr.UnknownValueError:
#         print("Speech recognition could not understand audio")
#         return ""

#     except sr.RequestError as e:
#         print(f"Error: {e}")
#         return ""

In [ ]:
p = pyaudio.PyAudio()
for i in range(p.get_device_count()):
    print(f"{i}: "+ p.get_device_info_by_index(i)["name"])
p.terminate()

In [24]:
# load the whisper model
whisper_model = None

def initialize_whisper():
    global whisper_model
    if whisper_model is None:
        print("Loading whisper model ...")
        whisper_model = whisper.load_model("medium")
        print("Whisper model loaded")

In [25]:
def recognize_speech(audio_data):
    try:
        initialize_whisper()
        # convert audio to NumPy array
        audio_bytes = audio_data.get_wav_data()
        audio_buffer = io.BytesIO(audio_bytes)
        sample_rate, audio_array = wavfile.read(audio_buffer)

        # preprocess audio
        if audio_array.dtype != np.float32:
            audio_array = audio_array.astype(np.float32) / 32768.0
        # reduce noise
        audio_array = reduce_noise(y=audio_array, sr =sample_rate)
        # normalize volume & trim silence moments
        audio_array = librosa.util.normalize(audio_array)
        audio_array, _ = librosa.effects.trim(audio_array, top_db=20)
        
        # save the output and transcibtion
        wavfile.write("test_recording.wav", sample_rate, audio_array)
        result = whisper_model.transcribe(audio_array, language="fa")
        text = result["text"].strip()
        print(f"Recognized Input Voice: {text}") if text else print("No speech detected")

    except Exception as e:
        print(f"Error: {e}")
        return ""

In [35]:
# Testing ...
init_response_AU = start_recording(RECORD_SECONDS=5)


Listening....


Thinking... 



In [36]:
init_response_TXT = recognize_speech(init_response_AU)

Recognized Input Voice: سلام. دارم صدا رو تسلي کنم. دیگر بیانم که ایکیت...


In [33]:
init_response_TXT

In [ ]:
# function to read the input text aloud
def speak(text, speed=180):
    # Text-To-Speech
    engine = pyttsx3.init()
    engine.setProperty('rate', speed)  # how fast it talks
    # just a customization on the narator voice
    voices = engine.getProperty('voices')
    engine.setProperty('voice', voices[0].id)  # Change index to select different voices
    engine.say(text)
    engine.runAndWait()

# speak(text="Hello Mohammad Hossein")


In [ ]:
def get_llm_response(text, model="deepseek/deepseek-chat:free"):
    """Interacts with DeepSeek model through OpenRouter API"""
    
    API_KEY = "#####"
    API_URL = 'https://openrouter.ai/api/v1/chat/completions'
    # required to make sure it bypasses the authentication
    headers = {
        'Authorization': f'Bearer {API_KEY}',
        'HTTP-Referer': 'https://github.com/MHosseinHashemi'
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": text}]
    }


    # its one hell of a exceptioon handling  considering the internet issues we have -------------------------------
    try:
        response = requests.post(API_URL, json=data, headers=headers)
        response.raise_for_status()  # Raises exception for 4XX/5XX errors
        return response.json()['choices'][0]['message']['content']

    
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return "Sorry, I couldn't process your request right now."

    
    except (KeyError, IndexError):
        print("Unexpected API response format")
        return "Sorry, I received an unexpected response."

        

In [ ]:
def talk_to_me(flag):
    begin_flag = 0
    # In case the camera is unplugged or mulfunctioned
    if flag == -1:
        speak("Sorry, It seems there is a problem with the camera, shutting down")
        return

    # Not authorized (face not detected as authrized)
    if flag == 0:
        speak("Intruder, I can See You", 125)
        return

    speak("Hello sir, How Can I help you?")
    begin_flag = 1

    while begin_flag == 1:
        try:
            # Record and recognize speech
            init_response_AU = start_recording(RECORD_SECONDS=10) # can dynamically set this via a prompt itself
            init_response_TXT = recognize_speech(init_response_AU)
            
            # Handle empty or failed recognition
            if not init_response_TXT or init_response_TXT.strip() == "":
                speak("Sorry, I didn't catch that. Please try again.")
                continue

            # print(f"Recognized: {init_response_TXT}")

            # Check for shutdown commands 
            # (This is not a reliable considering that the a prompt itself might include these words, but for now its ok)
            shutdown_keywords = ["shut down", "shutdown", "power off", "poweroff" "turn off", "turnoff" "close", "exit"]
            if any(keyword in init_response_TXT.lower() for keyword in shutdown_keywords):
                speak(f"{keyword} command received! Goodbye")
                begin_flag = 0
                break

            # Process the response with LLM and continue conversation
            llm_answer = get_llm_response(init_response_TXT)
            print(f"LLM Response: {llm_answer}")
            speak(llm_answer)
            time.sleep(0.5)  # Brief pause to avoid overlapping speech and recording


        except Exception as e:
            print(f"Error in speech recognition or processing: {e}")
            speak("An error occurred. Please try again.")
            continue

    return

# Now to catch the introder

We need a function that sends the image captured by the verify_the_face() to the telegram acount of authorized people via API
and warn them off the incident

In [ ]:
# Minimal beginnig
def main():
    # 1. See who is here ...
    flag = verify_the_face()
    # 2. Listen to what I have to say
    talk_to_me(flag)

In [ ]:
# main()

### Let's integrate the Telegram API

In [ ]:
# BOT_TOKEN = '####' # use bot father to get these two values
# CHAT_ID = '####'  

In [ ]:
def send_file_to_telegram(bot_token, chat_id, file_path, caption="Here is your file"):
    url = f"https://api.telegram.org/bot{bot_token}/sendDocument"
    with open(file_path, 'rb') as file:
        files = {'document': file}
        data = {'chat_id': chat_id, 'caption': caption}
        response = requests.post(url, files=files, data=data)
        
    # if response.status_code == 200:
        # print(f"File {file_path} sent successfully to chat ID {chat_id}")
    # else:
        # print(f"Error sending file: {response.json()}")


In [ ]:
## Configuration 
# BOT_TOKEN = "####"  
# CHAT_ID = "####"     
# FILE_PATH = r"C:\\Users\\laste\\Desktop\\Scene Understanding\\test.jpg"  

# send_file_to_telegram(BOT_TOKEN, CHAT_ID, FILE_PATH)

In [ ]:
# have to modify this function to integrate this new functionality
def verify_the_face(SIMILARITY_THRESHOLD=35,
                   camera_input_index=0,
                   weights="buffalo_l",
                   root="C:\\Users\\laste\\Desktop\\Scene Understanding\\face_verification_models\\.insightface\\",
                   reference_image="sample.jpg",
                   use_gpu=True, 
                   BOT_TOKEN = "####",  
                   CHAT_ID = "####" ):
    
    # Initialize model
    app = FaceAnalysis(name=weights, root=root)
    app.prepare(ctx_id=0 if use_gpu else -1, det_size=(640, 640))

    # Load reference face
    your_face_img = cv2.imread(reference_image)
    if your_face_img is None:
        print(f"Error: Reference image not found at {reference_image}")
        permission_flag = -1
        return permission_flag
        
    your_face = app.get(your_face_img)
    if not your_face:
        print("Error: No face detected in reference image")
        permission_flag = -1
        return permission_flag
    
    anchor_face_embedding = your_face[0].embedding
    permission_flag = 0
    found_you = False
    unknown_faces = set()

    cap = cv2.VideoCapture(camera_input_index)
    if not cap.isOpened():
        print(f"Error: Could not open camera {camera_input_index}")
        permission_flag = -1
        return permission_flag

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            faces = app.get(frame)
            if not faces:
                cv2.imshow("Face Verification", frame)
                permission_flag = -1
                if cv2.waitKey(1) == ord('q'):
                    break
                continue
                
            current_frame_face_ids = set()
            
            # for face in faces:
            face = faces[0]
            max_sim = (face.embedding @ face.embedding.T).item()
            similarity = (anchor_face_embedding @ face.embedding.T).item()
            similarity = (similarity * 100) / max_sim 
            x1, y1, x2, y2 = face.bbox.astype(int)
            face_id = f"{x1}_{y1}_{x2}_{y2}"
            current_frame_face_ids.add(face_id)
                
            if similarity >= SIMILARITY_THRESHOLD and not found_you:

                found_you = True
                permission_flag = 1
                color = (0, 255, 0)  # Green
                                        
                # Draw green box and show for 3 seconds
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.imshow("Face Verification", frame)
                cv2.waitKey(2000)
                break
    
            else:
                # if it not a verified person
                color = (0, 0, 255) 
                timestamp = datetime.now().strftime("%m%d_%H%M%S")
                permission_flag = 0

                filename = f"Unknown_Person_M{timestamp[0:2]}_D{timestamp[2:4]}_H{timestamp[-6:-4]}_M{timestamp[-4:-2]}_S{timestamp[-2:]}.jpg"
                time_log = f"Unknown Person Detected!\nMonth: {timestamp[0:2]}\nDay: {timestamp[2:4]}\nTime: {timestamp[-6:-4]}:{timestamp[-4:-2]}:{timestamp[-2:]}"
                # print(filename)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.imwrite(filename, frame)
                send_file_to_telegram(BOT_TOKEN, CHAT_ID, filename, time_log) # send the img to Admin Telegram
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.imshow("Face Verification", frame)
                cv2.waitKey(2000)
                break

            if found_you:
                break
                
                
    finally:
        cap.release()
        cv2.destroyAllWindows()
    
    return permission_flag

### putting it all together

In [ ]:
def main():
    # 1. See who is here ...
    flag = verify_the_face()
    # 2. Listen to what I have to say
    talk_to_me(flag)

In [ ]:
main()